<a href="https://colab.research.google.com/github/Shreya-1910/ArtificalMethodsCW/blob/Preprocessed-data/dataProc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import requests
import os
import zipfile
import pandas as pd

#Change path to where your files are saved
!unzip -n "/content/drive/MyDrive/AIMcw/MachineLearningCSV.zip"

file_name= os.path.join("MachineLearningCVE", "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
df_raw = pd.read_csv(file_name, skipinitialspace=True)

#loading all csv files from folder
def load_files(data_path, file_list):
  df_list = []
  for file in file_list:
      file_path = os.path.join(data_path, file)
      try:
          df = pd.read_csv(file_path, skipinitialspace=True, low_memory=False)
          df_list.append(df)
      except Exception as e:
          print(f" Error: {e}")

  if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"\nTOTAL: {len(combined_df)} rows, {combined_df.shape[1]} columns")
    return combined_df
  else:
    print("No files loaded")
    return None


data_path="MachineLearningCVE"

FILE_NAMES= ["Wednesday-workingHours.pcap_ISCX.csv",
              "Tuesday-WorkingHours.pcap_ISCX.csv",
              "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
              "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
              "Monday-WorkingHours.pcap_ISCX.csv",
              "Friday-WorkingHours-Morning.pcap_ISCX.csv",
              "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
              "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv" ]


df= load_files(data_path, FILE_NAMES)


Mounted at /content/drive
Archive:  /content/drive/MyDrive/AIMcw/MachineLearningCSV.zip
   creating: MachineLearningCVE/
  inflating: MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  

TOTAL: 2830743 rows, 79 columns


In [4]:
import requests
import zipfile
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pandas as pd

#Get list of original columns
original_cols= df.columns.tolist()
print(f"  Original columns: {len(original_cols)}")

#clean column names by removing special characters/spaces
df.columns= df.columns.str.strip()
df.columns= df.columns.str.replace(' ', '_')
df.columns= df.columns.str.replace('/','_')
df.columns= df.columns.str.replace('-','_')
df.columns = df.columns.str.replace('(', '')
df.columns = df.columns.str.replace(')', '')

#Remove duplicate rows if they exist
initial_rows=len(df)
duplicates= df.duplicated().sum()

if duplicates>0:
  df.drop_duplicates(inplace=True)


print(f"\nData types:")
print(df.dtypes.value_counts())


#Find label column
potential_label_cols = ['Label', 'label', 'Class', 'class']
label_col = None

for col in potential_label_cols:
  if col in df.columns:
    label_col = col
    break

if label_col is None:
  label_col= df.columns[-1]

print(f"\nLabel column: {label_col}")

unique_labels= df[label_col].unique()

#Fix incorrect labels

label_fixes= {
  "Web Attack � Brute Force": "Web Attack-Brute Force",
  "Web Attack � XSS": "Web Attack-XSS",
  "Web Attack � Sql Injection": "Web Attack-Sql Injection",
  'Web Attack  Brute Force': 'Web Attack-Brute Force',
  'Web Attack  XSS': 'Web Attack-XSS',
  'Web Attack  Sql Injection': 'Web Attack-Sql Injection',
  'Web Attack\x96 Brute Force': 'Web Attack-Brute Force',
  'Web Attack\x96 XSS': 'Web Attack-XSS',
  'Web Attack\x96 Sql Injection': 'Web Attack-Sql Injection'
}

before_labels = df[label_col].unique()
df[label_col] = df[label_col].replace(label_fixes)
after_labels = df[label_col].unique()

#Create encoded labels for binary classification
df['Label_Binary'] = (df[label_col] != 'BENIGN').astype(int)
label_encoder = LabelEncoder()
df['Label_Multi'] = label_encoder.fit_transform(df[label_col])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print(f"  Binary: 0 = BENIGN, 1 = Attack")
print(f"  Multi-class: {len(label_mapping)} classes")
print(f"  Sample mapping: {dict(list(label_mapping.items())[:5])}")

# Display class distribution
print(f"\n  Class distribution (binary):")
print(f"    BENIGN: {(df['Label_Binary'] == 0).sum():,}")
print(f"    ATTACK: {(df['Label_Binary'] == 1).sum():,}")
print(f"    Ratio: 1:{((df['Label_Binary'] == 0).sum() / (df['Label_Binary'] == 1).sum()):.2f}")


#Remove leaky features
leaks=[
      'flow id','flow_id','flowid',
      'source ip', 'src_ip', 'source_ip',
      'destination ip', 'dst_ip', 'dest_ip',
      'protocol',
      'timestamp', 'time_stamp', 'fwd id',
      'unix', 'stdt', 'stime', 'ltime'
  ]

leaky_features=[]

for col in df.columns:
  col_lower= col.lower()
  if any(term in col_lower for term in leaks):
    leaky_features.append(col)


protocol_cols=[col for col in leaky_features if 'protocol' in col.lower()]
cols_to_remove=[col for col in leaky_features if col not in protocol_cols]

if len(cols_to_remove) > 10:
    print(f"    ... and {len(cols_to_remove)-10} more")

if protocol_cols:
    print(f"\n  Keeping {len(protocol_cols)} protocol columns:")
    for col in protocol_cols:
        print(f"    - {col}")

df_no_leaks= df.drop(columns= cols_to_remove, errors='ignore')
print(f"\n  Features after removal: {df_no_leaks.shape[1]}")


#Select all columns as features except label columns
feature_cols=[ col for col in df_no_leaks.columns if col not in[label_col,'Label_Binary','Label_Multi']]

x= df_no_leaks[feature_cols]
y_binary= df_no_leaks['Label_Binary']
y_multi= df_no_leaks['Label_Multi']

print(f"  Features: {x.shape[1]} columns")
print(f"  Binary target: {len(y_binary.unique())} classes")
print(f"  Multi target: {len(y_multi.unique())} classes")

#Remove low variance features
variances= x.var()
constant_features= variances[variances==0].index.tolist()
low_variance_features= variances[variances<0.01].index.tolist()

print(f"  {len(constant_features)} constant features")
print(f"  {len(low_variance_features)} low variance features")

#Remove constant features
if constant_features:
  print(f"    - {constant_features}")
  for col in constant_features:
    print(f"      - {col}: {x[col].unique()}")
  x= x.drop(columns=constant_features)
  feature_cols= [col for col in feature_cols if col not in constant_features]
print(f"  Features after removal: {x.shape[1]} columns")

#Handle infinite values
inf_mask= np.isinf(x).any()
inf_cols=inf_mask[inf_mask].index.tolist()

if inf_cols:
  print(f"  {len(inf_cols)} infinite values")
  x= x.replace([np.inf, -np.inf], np.nan)


  Original columns: 79

Data types:
int64      54
float64    24
object      1
Name: count, dtype: int64

Label column: Label
  Binary: 0 = BENIGN, 1 = Attack
  Multi-class: 15 classes
  Sample mapping: {'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4)}

  Class distribution (binary):
    BENIGN: 2,096,484
    ATTACK: 425,878
    Ratio: 1:4.92

  Features after removal: 81
  Features: 78 columns
  Binary target: 2 classes
  Multi target: 15 classes
  8 constant features
  12 low variance features
    - ['Bwd_PSH_Flags', 'Bwd_URG_Flags', 'Fwd_Avg_Bytes_Bulk', 'Fwd_Avg_Packets_Bulk', 'Fwd_Avg_Bulk_Rate', 'Bwd_Avg_Bytes_Bulk', 'Bwd_Avg_Packets_Bulk', 'Bwd_Avg_Bulk_Rate']
      - Bwd_PSH_Flags: [0]
      - Bwd_URG_Flags: [0]
      - Fwd_Avg_Bytes_Bulk: [0]
      - Fwd_Avg_Packets_Bulk: [0]
      - Fwd_Avg_Bulk_Rate: [0]
      - Bwd_Avg_Bytes_Bulk: [0]
      - Bwd_Avg_Packets_Bulk: [0]
      - Bwd_Avg_Bulk_Rate: [0]
  Featu

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

#Split data
X_train, X_test, y_train, y_test= train_test_split(x, y_binary, test_size=0.2, random_state=42)

#Handle missing values
imputer= SimpleImputer(strategy='median')
X_train_imputed= imputer.fit_transform(X_train)
X_test_imputed= imputer.transform(X_test)

#Scale features
scaler= StandardScaler()
X_train_scaled= scaler.fit_transform(X_train_imputed)
X_test_scaled= scaler.fit_transform(X_test_imputed)


In [ ]:
#Saving data- change the paths

save_path= "/content/drive/MyDrive/AIMcw/preprocessed_data/"

!mkdir -p "{save_path}"
np.save(os.path.join(save_path, 'x_full.npy'), x.values)
np.save(os.path.join(save_path, 'y_binary_full.npy'), y_binary.values)
np.save(os.path.join(save_path, 'y_multi_full.npy'), y_multi.values)

if 'feature_cols' in dir():
  with open(os.path.join(save_path, 'feature_names.txt'),'w') as f:
    for name in feature_cols:
            f.write(f"{name}\n")

df_no_leaks.to_csv(f'{save_path}/preprocessed_data.csv', index=False)
x.to_csv(f'{save_path}/features.csv', index=False)
y_binary.to_csv(f'{save_path}/target_binary.csv', index=False, header=['Label_Binary'])
y_multi.to_csv(f'{save_path}/target_multi.csv', index=False, header=['Label_Multi'])